# EDA 01 — Raw metadata files exploration

This notebook performs an initial exploratory analysis of the raw metadata files used in the project.

The objective is to inspect the structure, dimensions, columns, missing values and basic content of the three original CSV files stored in `data/raw/metadata`.

No preprocessing or feature engineering is performed in this notebook. The goal is only to understand the raw input data before defining the preprocessing pipeline.

In [96]:
# Libraries
from skin_lesion_ai.utils.config import load_raw_metadata
import numpy as np
import pandas as pd

In [2]:
# Loading data
df1, df2, df3 = load_raw_metadata()
print("df1 - ground_truth:", df1.shape)
print("df2 - supplement:", df2.shape)
print("df3 - metadata:", df3.shape)

/Users/carlesraichbros/my-image-classifier/src/skin_lesion_ai/utils/config.py:43: DtypeWarning: Columns (0: iddx_5) have mixed types. Specify dtype option on import or set low_memory=False.
  supplement = pd.read_csv(path(raw["supplement_csv"]))


df1 - ground_truth: (401059, 2)
df2 - supplement: (401059, 13)
df3 - metadata: (401059, 42)


## ground_truth (df1)

In [3]:
# Data Wrangler
df1

,isic_id,malignant
0,ISIC_0015670,0.0
1,ISIC_0015845,0.0
2,ISIC_0015864,0.0
3,ISIC_0015902,0.0
4,ISIC_0024200,0.0
...,...,...
401054,ISIC_9999937,0.0
401055,ISIC_9999951,0.0
401056,ISIC_9999960,0.0
401057,ISIC_9999964,0.0


In [4]:
# malignant col
df1["malignant"].unique().tolist()

[0.0, 1.0]

In [5]:
# malignant proportions
malignant_summary = (
    df1["malignant"]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / x["count"].sum(), 2))
)

malignant_summary.loc["Total"] = [
    malignant_summary["count"].sum(),
    malignant_summary["percentage"].sum(),
]

malignant_summary

,count,percentage
malignant,,
0.0,400666.0,99.9
1.0,393.0,0.1
Total,401059.0,100.0


## ground_truth (df1) summary

Basic lesion labeling. 

- **Rows**: 401059. No duplicated rows.
- **Cols**: 2
- **Col names**: [isic_id, malignant]

    - ***isic_id***: unique id per lesion (pkey). 0 NA.  

    - ***malignant***: malignancy flag per lesion. 0 NA. Two values: 
        - 0 = not malignant (400,666 counts; 99.9%). 
        - 1 = malignant (393 counts; 0.1%).

## supplement (df2)

In [6]:
# Data Wrangler
df2

,isic_id,attribution,copyright_license,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,Memorial Sloan Kettering Cancer Center,CC-BY,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040
3,ISIC_0015902,ACEMID MIA,CC-0,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.989998
4,ISIC_0024200,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,70.442510
...,...,...,...,...,...,...,...,...,...,...,...,...,...
401054,ISIC_9999937,"Department of Dermatology, Hospital Clínic de ...",CC-BY-NC,IL_9520694,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999988
401055,ISIC_9999951,Memorial Sloan Kettering Cancer Center,CC-BY,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999820
401056,ISIC_9999960,"Frazer Institute, The University of Queensland...",CC-BY,IL_9852274,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.999416
401057,ISIC_9999964,University Hospital of Basel,CC-BY-NC,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,100.000000


In [7]:
df2.columns.tolist()

['isic_id',
 'attribution',
 'copyright_license',
 'lesion_id',
 'iddx_full',
 'iddx_1',
 'iddx_2',
 'iddx_3',
 'iddx_4',
 'iddx_5',
 'mel_mitotic_index',
 'mel_thick_mm',
 'tbp_lv_dnn_lesion_confidence']

In [8]:
# isic_id col
isic_id_df1 = set(df1["isic_id"])
isic_id_df2 = set(df2["isic_id"])

print("Only in df1:", isic_id_df1 - isic_id_df2)
print("Only in df2:", isic_id_df2 - isic_id_df1)

Only in df1: set()
Only in df2: set()


In [9]:
# attribution col
df2["attribution"].unique().tolist()

['Memorial Sloan Kettering Cancer Center',
 'ACEMID MIA',
 'Department of Dermatology, Hospital Clínic de Barcelona',
 'University Hospital of Basel',
 'Frazer Institute, The University of Queensland, Dermatology Research Centre',
 'Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris',
 'ViDIR Group, Department of Dermatology, Medical University of Vienna']

In [10]:
# attribution proportions
attribution_summary = (
    df2["attribution"]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / x["count"].sum(), 2))
)

attribution_summary.loc["Total"] = [
    attribution_summary["count"].sum(),
    attribution_summary["percentage"].sum(),
]

attribution_summary

,count,percentage
attribution,,
Memorial Sloan Kettering Cancer Center,129068.0,32.18
"Department of Dermatology, Hospital Clínic de Barcelona",105724.0,26.36
University Hospital of Basel,65218.0,16.26
"Frazer Institute, The University of Queensland, Dermatology Research Centre",51768.0,12.91
ACEMID MIA,28665.0,7.15
"ViDIR Group, Department of Dermatology, Medical University of Vienna",12640.0,3.15
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",7976.0,1.99
Total,401059.0,100.00


In [11]:
# copyright_licese col
df2["copyright_license"].unique().tolist()

['CC-BY', 'CC-0', 'CC-BY-NC']

In [12]:
# license per site
pd.crosstab(df2["attribution"], df2["copyright_license"])

copyright_license,CC-0,CC-BY,CC-BY-NC
attribution,,,
ACEMID MIA,28665,0,0
"Department of Dermatology, Hospital Clínic de Barcelona",0,0,105724
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",0,7976,0
"Frazer Institute, The University of Queensland, Dermatology Research Centre",0,51768,0
Memorial Sloan Kettering Cancer Center,0,129068,0
University Hospital of Basel,0,0,65218
"ViDIR Group, Department of Dermatology, Medical University of Vienna",0,0,12640


In [13]:
# lesion_id col

n_null = df2["lesion_id"].isna().sum()
n_non_null = df2["lesion_id"].notna().sum()
n_dup = df2.loc[df2["lesion_id"].notna(), "lesion_id"].duplicated().sum()

lesion_id_summary = pd.DataFrame(
    {
        "count": [n_null, n_non_null, len(df2), n_dup],
        "percentage": [
            round(100 * n_null / len(df2), 2),
            round(100 * n_non_null / len(df2), 2),
            100.0,
            round(100 * n_dup / n_non_null, 2),
        ],
    },
    index=["Null", "Non-null", "Total", "Non-null duplicated"],
)

lesion_id_summary

,count,percentage
Null,379001,94.5
Non-null,22058,5.5
Total,401059,100.0
Non-null duplicated,0,0.0


In [14]:
# lesion_id per site
(
    df2.assign(has_lesion_id=df2["lesion_id"].notna())
    .groupby("attribution")["has_lesion_id"]
    .agg(total="size", with_lesion_id="sum")
    .assign(percentage=lambda x: round(100 * x["with_lesion_id"] / x["total"], 2))
    .sort_values("percentage", ascending=False)
)

,total,with_lesion_id,percentage
attribution,,,
"Frazer Institute, The University of Queensland, Dermatology Research Centre",51768,9056,17.49
ACEMID MIA,28665,1792,6.25
Memorial Sloan Kettering Cancer Center,129068,6212,4.81
University Hospital of Basel,65218,3004,4.61
"Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris",7976,167,2.09
"Department of Dermatology, Hospital Clínic de Barcelona",105724,1720,1.63
"ViDIR Group, Department of Dermatology, Medical University of Vienna",12640,107,0.85


In [15]:
# iddx_* cols

# iddx_1 col
iddx_1_summary = (
    df2["iddx_1"]
    .value_counts(dropna=False)
    .rename_axis("iddx_1")
    .reset_index(name="count")
)

iddx_1_summary["percentage"] = round(100 * iddx_1_summary["count"] / len(df2), 2)

iddx_1_summary

,iddx_1,count,percentage
0,Benign,400552,99.87
1,Malignant,393,0.10
2,Indeterminate,114,0.03


In [16]:
# iddx_2 col
iddx_2_summary = (
    df2["iddx_2"]
    .value_counts(dropna=False)
    .rename_axis("iddx_2")
    .reset_index(name="count")
)

iddx_2_summary["percentage"] = round(100 * iddx_2_summary["count"] / len(df2), 2)

iddx_2_summary

,iddx_2,count,percentage
0,NaN,399991,99.73
1,Benign melanocytic proliferations,443,0.11
2,Malignant adnexal epithelial proliferations - ...,163,0.04
3,Malignant melanocytic proliferations (Melanoma),157,0.04
4,Benign epidermal proliferations,83,0.02
5,Indeterminate melanocytic proliferations,75,0.02
6,Malignant epidermal proliferations,73,0.02
7,Indeterminate epidermal proliferations,39,0.01
8,Benign soft tissue proliferations - Fibro-hist...,15,0.00
9,Inflammatory or infectious diseases,7,0.00


In [17]:
for lvl1 in sorted(df2["iddx_1"].dropna().unique()):
    print(f"\n{lvl1}")

    children = df2.loc[df2["iddx_1"] == lvl1, "iddx_2"].dropna().value_counts()

    for child, n in children.items():
        print(f"  └─ {child}: {n}")


Benign
  └─ Benign melanocytic proliferations: 443
  └─ Benign epidermal proliferations: 83
  └─ Benign soft tissue proliferations - Fibro-histiocytic: 15
  └─ Inflammatory or infectious diseases: 7
  └─ Flat melanotic pigmentations - not melanocytic nevus: 5
  └─ Benign soft tissue proliferations - Vascular: 3
  └─ Cysts: 2
  └─ Benign adnexal epithelial proliferations - Follicular: 2
  └─ Benign adnexal epithelial proliferations - Apocrine or Eccrine: 1

Indeterminate
  └─ Indeterminate melanocytic proliferations: 75
  └─ Indeterminate epidermal proliferations: 39

Malignant
  └─ Malignant adnexal epithelial proliferations - Follicular: 163
  └─ Malignant melanocytic proliferations (Melanoma): 157
  └─ Malignant epidermal proliferations: 73


In [18]:
(
    df2.loc[df2["iddx_2"].isna(), "iddx_1"]
    .value_counts(dropna=False)
    .to_frame("count")
    .assign(percentage=lambda x: round(100 * x["count"] / len(df2), 2))
)

,count,percentage
iddx_1,,
Benign,399991,99.73


In [ ]:
# null iddx_2 and benign iddx_1
(df2.loc[df2["iddx_2"].isna(), "iddx_1"] == "Benign").all()

np.True_

In [20]:
# iddx_3 col
iddx_3_summary = (
    df2["iddx_3"]
    .value_counts(dropna=False)
    .rename_axis("iddx_3")
    .reset_index(name="count")
)

iddx_3_summary["percentage"] = round(100 * iddx_3_summary["count"] / len(df2), 2)

iddx_3_summary

,iddx_3,count,percentage
0,NaN,399994,99.73
1,Nevus,443,0.11
2,Basal cell carcinoma,163,0.04
3,Melanoma in situ,80,0.02
4,Atypical melanocytic neoplasm,64,0.02
5,Melanoma Invasive,63,0.02
6,Seborrheic keratosis,57,0.01
7,Squamous cell carcinoma in situ,49,0.01
8,Solar or actinic keratosis,39,0.01
9,"Squamous cell carcinoma, Invasive",22,0.01


In [21]:
# iddx_4 col
iddx_4_summary = (
    df2["iddx_4"]
    .value_counts(dropna=False)
    .rename_axis("iddx_4")
    .reset_index(name="count")
)

iddx_4_summary["percentage"] = round(100 * iddx_4_summary["count"] / len(df2), 2)

iddx_4_summary

,iddx_4,count,percentage
0,NaN,400508,99.86
1,"Nevus, Atypical, Dysplastic, or Clark",228,0.06
2,"Basal cell carcinoma, Nodular",98,0.02
3,"Basal cell carcinoma, Superficial",48,0.01
4,"Melanoma Invasive, Superficial spreading",37,0.01
5,"Nevus, NOS, Compound",30,0.01
6,"Nevus, NOS, Dermal",20,0.00
7,"Melanoma in situ, Lentigo maligna type",12,0.00
8,"Melanoma in situ, associated with a nevus",12,0.00
9,"Nevus, NOS, Junctional",10,0.00


In [22]:
# iddx_5 col
iddx_5_summary = (
    df2["iddx_5"]
    .value_counts(dropna=False)
    .rename_axis("iddx_5")
    .reset_index(name="count")
)

iddx_5_summary["percentage"] = round(100 * iddx_5_summary["count"] / len(df2), 2)

iddx_5_summary

,iddx_5,count,percentage
0,NaN,401058,100.0
1,"Blue nevus, Cellular",1,0.0


In [23]:
iddx_cols = ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"]

iddx_hierarchy_summary = (
    df2.groupby(iddx_cols, dropna=False).size().reset_index(name="count")
)

iddx_hierarchy_summary["percentage"] = round(
    100 * iddx_hierarchy_summary["count"] / len(df2), 4
)

iddx_hierarchy_summary = iddx_hierarchy_summary.sort_values(
    ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"], na_position="first"
)

iddx_hierarchy_summary

,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,count,percentage
27,Benign,NaN,NaN,NaN,NaN,399991,99.7337
0,Benign,Benign adnexal epithelial proliferations - Apo...,Hidradenoma,NaN,NaN,1,0.0002
1,Benign,Benign adnexal epithelial proliferations - Fol...,NaN,NaN,NaN,2,0.0005
2,Benign,Benign epidermal proliferations,Lichen planus like keratosis,NaN,NaN,11,0.0027
3,Benign,Benign epidermal proliferations,Pigmented benign keratosis,NaN,NaN,3,0.0007
5,Benign,Benign epidermal proliferations,Seborrheic keratosis,NaN,NaN,56,0.0140
4,Benign,Benign epidermal proliferations,Seborrheic keratosis,"Seborrheic keratosis, Clonal",NaN,1,0.0002
6,Benign,Benign epidermal proliferations,Solar lentigo,NaN,NaN,12,0.0030
17,Benign,Benign melanocytic proliferations,Nevus,NaN,NaN,141,0.0352
7,Benign,Benign melanocytic proliferations,Nevus,Blue nevus,"Blue nevus, Cellular",1,0.0002


In [24]:
# iddx_full col

iddx_cols = ["iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5"]
df2["iddx_reconstructed"] = (
    df2[iddx_cols]
    .fillna("")
    .apply(lambda x: "::".join([v for v in x if v != ""]), axis=1)
)

(df2["iddx_full"] == df2["iddx_reconstructed"]).value_counts()

True    401059
Name: count, dtype: int64

In [43]:
# merge df1 and df2
df12 = df1.merge(df2, on="isic_id", how="inner", validate="one_to_one")

print(df12.shape)

(401059, 15)


In [45]:
malignant_vs_iddx1 = pd.crosstab(df12["malignant"], df12["iddx_1"], margins=True)

malignant_vs_iddx1

iddx_1,Benign,Indeterminate,Malignant,All
malignant,,,,
0.0,400552,114,0,400666
1.0,0,0,393,393
All,400552,114,393,401059


In [46]:
malignant_vs_iddx2 = pd.crosstab(
    df12["malignant"], df12["iddx_2"], margins=True, dropna=False
)

malignant_vs_iddx2

iddx_2,Benign adnexal epithelial proliferations - Apocrine or Eccrine,Benign adnexal epithelial proliferations - Follicular,Benign epidermal proliferations,Benign melanocytic proliferations,Benign soft tissue proliferations - Fibro-histiocytic,Benign soft tissue proliferations - Vascular,Cysts,Flat melanotic pigmentations - not melanocytic nevus,Indeterminate epidermal proliferations,Indeterminate melanocytic proliferations,Inflammatory or infectious diseases,Malignant adnexal epithelial proliferations - Follicular,Malignant epidermal proliferations,Malignant melanocytic proliferations (Melanoma),NaN,All
malignant,,,,,,,,,,,,,,,,
0.0,1,2,83,443,15,3,2,5,39,75,7,0,0,0,399991,400666
1.0,0,0,0,0,0,0,0,0,0,0,0,163,73,157,0,393
All,1,2,83,443,15,3,2,5,39,75,7,163,73,157,399991,401059


In [ ]:
# manual tag and iddx_2 col check 1

manual_iddx_2_summary = (
    df2.loc[df2["lesion_id"].notna(), "iddx_2"]
    .value_counts(dropna=False)
    .rename_axis("iddx_2")
    .reset_index(name="count")
)

manual_iddx_2_summary["percentage_manual_tag"] = round(
    100 * manual_iddx_2_summary["count"] / df2["lesion_id"].notna().sum(), 2
)

manual_iddx_2_summary

,iddx_2,count,percentage_manual_tag
0,NaN,20990,95.16
1,Benign melanocytic proliferations,443,2.01
2,Malignant adnexal epithelial proliferations - ...,163,0.74
3,Malignant melanocytic proliferations (Melanoma),157,0.71
4,Benign epidermal proliferations,83,0.38
5,Indeterminate melanocytic proliferations,75,0.34
6,Malignant epidermal proliferations,73,0.33
7,Indeterminate epidermal proliferations,39,0.18
8,Benign soft tissue proliferations - Fibro-hist...,15,0.07
9,Inflammatory or infectious diseases,7,0.03


In [ ]:
#  manual tags vs iddx_2 null / non-null
has_manual_tag = df2["lesion_id"].notna()
iddx_2_status = (
    df2["iddx_2"]
    .notna()
    .map(
        {
            True: "iddx_2 non-null",
            False: "iddx_2 null",
        }
    )
)

manual_by_iddx_2_status = (
    df2.assign(
        has_manual_tag=has_manual_tag,
        iddx_2_status=iddx_2_status,
    )
    .groupby("iddx_2_status")
    .agg(
        total_cases=("isic_id", "count"),
        manual_tag_cases=("has_manual_tag", "sum"),
    )
    .assign(
        percentage_manual_tag=lambda x: round(
            100 * x["manual_tag_cases"] / x["total_cases"], 2
        )
    )
)

manual_by_iddx_2_status

,total_cases,manual_tag_cases,percentage_manual_tag
iddx_2_status,,,
iddx_2 non-null,1068,1068,100.00
iddx_2 null,399991,20990,5.25


In [28]:
# manual tags vs iddx_1 in non null iddx_2
manual_by_iddx_1_within_non_null_iddx_2 = (
    df2.loc[df2["iddx_2"].notna()]
    .assign(has_manual_tag=lambda x: x["lesion_id"].notna())
    .groupby("iddx_1")
    .agg(
        total_cases=("isic_id", "count"),
        manual_tag_cases=("has_manual_tag", "sum"),
    )
    .assign(
        percentage_manual_tag=lambda x: round(
            100 * x["manual_tag_cases"] / x["total_cases"], 2
        ),
    )
    .sort_values("manual_tag_cases", ascending=False)
)

manual_by_iddx_1_within_non_null_iddx_2

,total_cases,manual_tag_cases,percentage_manual_tag
iddx_1,,,
Benign,561,561,100.0
Malignant,393,393,100.0
Indeterminate,114,114,100.0


In [31]:
# mel_thick_mm and mel_mitotic_index cols
for col in ["mel_thick_mm", "mel_mitotic_index"]:
    print(f"\n=== {col} ===")

    display(
        df2.loc[df2[col].notna(), "iddx_full"]
        .value_counts(dropna=False)
        .to_frame("count")
    )


=== mel_thick_mm ===


,count
iddx_full,
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Superficial spreading",37
Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive,13
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Associated with a nevus",7
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, On chronically sun-exposed skin or lentigo maligna melanoma",5
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Nodular",1



=== mel_mitotic_index ===


,count
iddx_full,
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Superficial spreading",35
Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive,7
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Associated with a nevus",5
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, On chronically sun-exposed skin or lentigo maligna melanoma",5
"Malignant::Malignant melanocytic proliferations (Melanoma)::Melanoma Invasive::Melanoma Invasive, Nodular",1


In [35]:
for col in ["mel_thick_mm", "mel_mitotic_index"]:
    print(f"\n=== {col} ===")

    summary = pd.DataFrame(
        {
            "count": [df2[col].count()],
            "missing": [df2[col].isna().sum()],
            "missing_%": [round(100 * df2[col].isna().sum() / len(df2), 4)],
        }
    )

    display(summary)

    display(df2[col].describe())


=== mel_thick_mm ===


,count,missing,missing_%
0,63,400996,99.9843


count    63.000000
mean      0.670952
std       0.792798
min       0.200000
25%       0.300000
50%       0.400000
75%       0.600000
max       5.000000
Name: mel_thick_mm, dtype: float64


=== mel_mitotic_index ===


,count,missing,missing_%
0,53,401006,99.9868


count         53
unique         7
top       0/mm^2
freq          22
Name: mel_mitotic_index, dtype: object

In [ ]:
# tbp_lv_dnn_lesion_confidence col (lesion confidence)

df2["tbp_lv_dnn_lesion_confidence"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

count    4.010590e+05
mean     9.716220e+01
std      8.995782e+00
min      1.261082e-16
1%       5.542216e+01
5%       7.852168e+01
25%      9.966882e+01
50%      9.999459e+01
75%      9.999996e+01
95%      1.000000e+02
99%      1.000000e+02
max      1.000000e+02
Name: tbp_lv_dnn_lesion_confidence, dtype: float64

In [ ]:
# lesion confidence vs iddx_1
(df2.groupby("iddx_1")["tbp_lv_dnn_lesion_confidence"].describe())

,count,mean,std,min,25%,50%,75%,max
iddx_1,,,,,,,,
Benign,400552.0,97.180388,8.910078,1.261082e-16,99.670264,99.994610,99.999960,100.0
Indeterminate,114.0,87.500639,29.704617,6.865393e-08,99.057932,99.988927,99.999980,100.0
Malignant,393.0,81.431493,33.805649,2.256579e-06,83.221790,99.684890,99.995804,100.0


In [40]:
# lesion confidence vs iddx_2
(
    df2.loc[df2["iddx_2"].notna()]
    .groupby("iddx_2")["tbp_lv_dnn_lesion_confidence"]
    .describe()
    .sort_values("mean", ascending=False)
)

,count,mean,std,min,25%,50%,75%,max
iddx_2,,,,,,,,
Benign adnexal epithelial proliferations - Apocrine or Eccrine,1.0,99.999452,NaN,9.999945e+01,99.999452,99.999452,99.999452,99.999452
Benign adnexal epithelial proliferations - Follicular,2.0,99.708015,0.412930,9.941603e+01,99.562022,99.708015,99.854007,100.000000
Indeterminate melanocytic proliferations,75.0,96.895410,14.955200,1.560452e+01,99.975859,99.999809,100.000000,100.000000
Benign melanocytic proliferations,443.0,95.638540,17.780352,5.548416e-09,99.986955,99.999870,100.000000,100.000000
Flat melanotic pigmentations - not melanocytic nevus,5.0,92.276946,11.506631,7.383386e+01,88.043220,99.546885,99.961310,99.999450
Malignant melanocytic proliferations (Melanoma),157.0,87.114046,30.717087,2.255627e-05,99.112090,99.976740,99.999690,100.000000
Malignant adnexal epithelial proliferations - Follicular,163.0,82.903924,31.831764,3.286650e-06,88.965660,99.348181,99.988873,100.000000
Benign epidermal proliferations,83.0,82.760935,32.699677,6.066522e-05,81.150359,99.876201,99.995571,100.000000
Cysts,2.0,82.038397,24.862577,6.445790e+01,73.248148,82.038397,90.828645,99.618894


## supplement (df2) summary

Detailed lesion labeling.

- **Rows**: 401059. No duplicated rows.
- **Cols**: 13.
- **Col names**: ['isic_id','attribution','copyright_license','lesion_id','iddx_full','iddx_1''iddx_2','iddx_3','iddx_4','iddx_5','mel_mitotic_index','mel_thick_mm', 'tbp_lv_dnn_lesion_confidence'].

   - ***isic_id***: unique id per lesion (pkey). 0 NA. 100% match with df1.

   - ***attribution***: lesion site of origin. 0 NA. 7 sites (DESC order):

      - Memorial Sloan Kettering Cancer Center (USA): 129,086 lesions (32.18%).
      - Department of Dermatology, Hospital Clínic de Barcelona (Spain): 105,274 lesions (26.36%).
      - University Hospital of Basel (Switzerland): 65,218 lesions (16.26%).
      - Frazer Institute, The University of Queensland, Dermatology Research Centre (Australia): 65,218 lesions (12.91%).
      - ACEMID MIA (Australian Centre of Excellence in Melanoma Imaging and Diagnosis - Melanoma Institute Australia), which includes Alfred Hospital and FNQH Cairn (Australia): 28,665 lesions (7.15%).
      - ViDIR Group, Department of Dermatology, Medical University of Vienna (Austria): 12,640 lesions (3.15%).
      - Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris (Greece): 7,976 lesions (1.99%).

   - ***copyright_license***: 3 types of copyright license. 0 NA.

      - ***CC-0 (Creative Commons Zero)***: Public domain dedication. The data can be used, modified, and redistributed without attribution and without restrictions. This applies to:
         - ACEMID MIA

      - ***CC-BY (Creative Commons Attribution)***: The data can be used, modified, and redistributed, including for commercial purposes, provided that appropriate attribution is given to the original source. This applies to:
         - Memorial Sloan Kettering Cancer Center
         - Frazer Institute, The University of Queensland, Dermatology Research Centre
         - Department of Dermatology, University of Athens, Andreas Syggros Hospital of Skin and Venereal Diseases, Alexander Stratigos, Konstantinos Liopyris

      - ***CC-BY-NC (Creative Commons Attribution–NonCommercial)***: The data can be used, modified, and redistributed with attribution, but commercial use is not permitted. This applies to:
         - Department of Dermatology, Hospital Clínic de Barcelona
         - University Hospital of Basel
         - ViDIR Group, Department of Dermatology, Medical University of Vienna

   - ***lesion_id***: id for some lesions indicating it was a manual tag. Non null and unique for 20,058 cases (5.5%), the rest of lesions have a null lesion_id. All sites present manual tags.

   - ***iddx_full / iddx_1-5***: hierarchical lesion diagnosis labels.

      The diagnosis variables follow a hierarchical taxonomy of increasing specificity. `iddx_1` contains the broadest diagnostic category, while `iddx_5` contains the most specific diagnosis when available. `iddx_full` stores the complete hierarchical path as a single string.

      Three first-level (`iddx_1`) diagnostic groups are present:

      - *Benign*: 400,552 lesions (99.87%).
      - *Malignant*: 393 lesions (0.10%).
      - *Indeterminate*: 114 lesions (0.03%).

      The second-level diagnosis (`iddx_2`) contains the following categories:

      - ***Benign iddx_1***:
         - NULL (no biopsy): 399,991 lesions (99.73%).
         - Benign melanocytic proliferations: 443 lesions (0.11%).
         - Benign epidermal proliferations: 83 lesions (0.02%).
         - Benign soft tissue proliferations - Fibro-histiocytic: 15 lesions (<0.01%).
         - Inflammatory or infectious diseases: 7 lesions (<0.01%).
         - Flat melanotic pigmentations - not melanocytic nevus: 5 lesions (<0.01%).
         - Benign soft tissue proliferations - Vascular: 3 lesions (<0.01%).
         - Cysts: 2 lesions (<0.01%).
         - Benign adnexal epithelial proliferations - Follicular: 2 lesions (<0.01%).
         - Benign adnexal epithelial proliferations - Apocrine or Eccrine: 1 lesion (<0.01%).

      - ***Indeterminate iddx_1***:
         - Indeterminate melanocytic proliferations: 75 lesions (0.02%).
         - Indeterminate epidermal proliferations: 39 lesions (0.01%).

      - ***Malignant iddx_1***:
         - Malignant adnexal epithelial proliferations - Follicular: 163 lesions (0.04%).
         - Malignant melanocytic proliferations (Melanoma): 157 lesions (0.04%).
         - Malignant epidermal proliferations: 73 lesions (0.02%).

      The deeper diagnostic levels (`iddx_3-5`) further refine the diagnosis hierarchy for some non null `iddx_2` lesions into increasingly specific entities and histopathological subtypes (see `iddx_hierarchy_summary`). Across the dataset, 52 unique diagnostic paths are observed. 
      
      Only 1,068 lesions (0.27% of the dataset) contain a biopsy-confirmed detailed diagnosis beyond the first diagnostic level (`iddx_2-5`). Among these biopsied lesions, the five most frequent second-level diagnostic categories are:

      - Benign melanocytic proliferations: 443 lesions (41.48%).
      - Malignant adnexal epithelial proliferations - Follicular: 163 lesions (15.26%).
      - Malignant melanocytic proliferations (Melanoma): 157 lesions (14.70%).
      - Benign epidermal proliferations: 83 lesions (7.77%).
      - Indeterminate melanocytic proliferations: 75 lesions (7.02%).

      Together, these categories account for 86.24% of all biopsied lesions. 
      
      Relationship between lesion_ic and iddx_*

      An important observation is that all lesions with detailed diagnostic information (`iddx_2-5`) are associated with a non-null `lesion_id`. In other words, 100% of lesions with a detailed diagnosis belong to the manually tagged subset of the dataset. Conversely, among lesions lacking a detailed diagnosis (`iddx_2 = NA`), only 20,990 cases (5.25%) contain a manual lesion identifier. This suggests a strong association between manual annotation and the availability of detailed diagnostic information.

      Relationship between ground truth and iddx_*

      Perfect correspondence between the binary target and the first diagnostic level (`iddx_1`). Lesions classified as *Malignant* in the diagnostic hierarchy are labelled as malignant ('1') in df1, while lesions classified as *Benign*  are labelled as non-malignant ('0').

      Important remark: all lesions classifies as *Indeterminate* are labelled as non-malignant ('0') too.

   - ***mel_thick_mm***: thickness in depth of melanoma invasion (Breslow thickness, in millimetres) derived from histopathological examination (lesions that underwent biopsy), measurement to assess melanoma severity. 

      The variable is available for only 63 melanoma lesions (approximately 40.1% of all melanoma cases). The remaining 400,996 observations (99.984% of the dataset) are missing.

      All non-null observations correspond to invasive melanoma diagnoses. Values range from 0.2 mm to 5.0 mm, with a median thickness of 0.4 mm (IQR: 0.3–0.6 mm). The distribution is right-skewed, with a small number of thicker lesions reaching up to 5 mm.

   - ***mel_mitotic_index***: mitotic index of invasive malignant melanomas derived from histopathological examination (lesions that underwent biopsy), measurement to assess melanoma severity. 

      The variable is available for only 53 melanoma lesions (approximately 33.8% of all melanoma cases). The remaining 401,006 observations (99.987% of the dataset) are missing.

      All non-null observations correspond to invasive melanoma diagnoses. The variable is stored as a categorical measurement (e.g. "0/mm²") and contains seven distinct values. The most frequent category is 0/mm², representing 22 of the 53 recorded cases (41.5%).

      Together, `mel_thick_mm` and `mel_mitotic_index` provide histopathological information that is only available for a very small subset of invasive melanomas within the dataset.

   - ***tbp_lv_dnn_lesion_confidence***: lesion confidence score on a 0–100 scale. Unspecified / unknown (probably internal) score. 0 NA.

      Values range from 0 to 100, with a mean of 97.16 and a median of 99.99. The distribution is heavily concentrated near the upper bound of the scale, with 75% of lesions having scores above 99.999 and 95% above 100.

      This indicates that the vast majority of lesions were assigned a very high confidence score. Based on its distribution and the dataset documentation, the variable appears to represent confidence in lesion detection rather than a direct measure of malignancy risk.


## metadata (df3)

In [47]:
# Data Wrangler
df3

,isic_id,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,...,tbp_lv_norm_color,tbp_lv_perimeterMM,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z
0,ISIC_0015670,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,16.261975,...,0.000000,9.307003,0.000000,2.036195,2.637780,0.590476,85,-182.703552,613.493652,-42.427948
1,ISIC_0015845,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,25.364740,...,0.000000,3.354148,0.000000,0.853227,3.912844,0.285714,55,-0.078308,1575.687000,57.174500
2,ISIC_0015864,IP_6724798,60.0,male,posterior torso,3.40,TBP tile: close-up,3D: XP,22.575830,17.128170,...,0.000000,8.886309,0.000000,1.743651,1.950777,0.361905,105,123.649700,1472.010000,232.908900
3,ISIC_0015902,IP_4111386,65.0,male,anterior torso,3.22,TBP tile: close-up,3D: XP,14.242329,12.164757,...,1.771705,9.514499,0.664690,1.258541,1.573733,0.209581,130,-141.024780,1442.185791,58.359802
4,ISIC_0024200,IP_8313778,55.0,male,anterior torso,2.73,TBP tile: close-up,3D: white,24.725520,20.057470,...,0.000000,6.467562,0.000000,2.085409,2.480509,0.313433,20,-72.315640,1488.720000,21.428960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401054,ISIC_9999937,IP_1140263,70.0,male,anterior torso,6.80,TBP tile: close-up,3D: XP,22.574335,14.944666,...,7.348126,20.210836,2.328066,7.054819,2.169398,0.288920,100,147.187256,1182.317505,122.652588
401055,ISIC_9999951,IP_5678181,60.0,male,posterior torso,3.11,TBP tile: close-up,3D: white,19.977640,16.026870,...,2.795246,9.340242,1.048147,1.879502,2.910780,0.460000,25,52.349740,1393.187000,127.261700
401056,ISIC_9999960,IP_0076153,65.0,female,anterior torso,2.05,TBP tile: close-up,3D: XP,17.332567,12.364397,...,1.660411,5.999862,0.607554,1.702824,2.205272,0.183099,40,54.622246,1065.263672,-106.833740
401057,ISIC_9999964,IP_5231513,30.0,female,anterior torso,2.80,TBP tile: close-up,3D: XP,22.288570,9.564721,...,3.583966,9.113276,1.078204,3.680175,1.957157,0.161850,140,-9.861557,877.527000,-76.982120


In [48]:
# metadata cols
df3.columns.tolist()

['isic_id',
 'patient_id',
 'age_approx',
 'sex',
 'anatom_site_general',
 'clin_size_long_diam_mm',
 'image_type',
 'tbp_tile_type',
 'tbp_lv_A',
 'tbp_lv_Aext',
 'tbp_lv_B',
 'tbp_lv_Bext',
 'tbp_lv_C',
 'tbp_lv_Cext',
 'tbp_lv_H',
 'tbp_lv_Hext',
 'tbp_lv_L',
 'tbp_lv_Lext',
 'tbp_lv_areaMM2',
 'tbp_lv_area_perim_ratio',
 'tbp_lv_color_std_mean',
 'tbp_lv_deltaA',
 'tbp_lv_deltaB',
 'tbp_lv_deltaL',
 'tbp_lv_deltaLB',
 'tbp_lv_deltaLBnorm',
 'tbp_lv_eccentricity',
 'tbp_lv_location',
 'tbp_lv_location_simple',
 'tbp_lv_minorAxisMM',
 'tbp_lv_nevi_confidence',
 'tbp_lv_norm_border',
 'tbp_lv_norm_color',
 'tbp_lv_perimeterMM',
 'tbp_lv_radial_color_std_max',
 'tbp_lv_stdL',
 'tbp_lv_stdLExt',
 'tbp_lv_symm_2axis',
 'tbp_lv_symm_2axis_angle',
 'tbp_lv_x',
 'tbp_lv_y',
 'tbp_lv_z']

In [52]:
# isic_id col
isic_id_df1 = set(df1["isic_id"])
isic_id_df3 = set(df3["isic_id"])

print("Only in df1:", isic_id_df1 - isic_id_df3)
print("Only in df3:", isic_id_df3 - isic_id_df1)

Only in df1: set()
Only in df3: set()


In [ ]:
# patient_id

# n_lesions per patient
lesions_per_patient = df3.groupby("patient_id").size().reset_index(name="n_lesions")

lesions_per_patient["n_lesions"].describe()

count    1042.000000
mean      384.893474
std       540.268913
min         1.000000
25%       115.000000
50%       241.500000
75%       477.500000
max      9184.000000
Name: n_lesions, dtype: float64

In [ ]:
# n_malignant_lesions (df1) per patient
patient_malignancy = df3[["isic_id", "patient_id"]].merge(
    df1[["isic_id", "malignant"]], on="isic_id", how="inner"
)

malignant_per_patient = patient_malignancy.groupby("patient_id")["malignant"].sum()

patient_malignancy_summary = (
    malignant_per_patient.value_counts()
    .sort_index()
    .rename_axis("n_malignant_lesions")
    .reset_index(name="count")
)

patient_malignancy_summary["percentage"] = round(
    100
    * patient_malignancy_summary["count"]
    / patient_malignancy_summary["count"].sum(),
    3,
)

patient_malignancy_summary["cumulative_percentage"] = round(
    patient_malignancy_summary["percentage"].cumsum(), 3
)

n_patients_with_malignancy = malignant_per_patient.gt(0).sum()

patient_malignancy_summary["percentage_within_malignant_patients"] = round(
    100 * patient_malignancy_summary["count"] / n_patients_with_malignancy, 3
)

patient_malignancy_summary.loc[
    patient_malignancy_summary["n_malignant_lesions"] == 0,
    "percentage_within_malignant_patients",
] = None

patient_malignancy_summary.loc[len(patient_malignancy_summary)] = [
    "Total",
    patient_malignancy_summary["count"].sum(),
    round(patient_malignancy_summary["percentage"].sum(), 3),
    None,
    round(patient_malignancy_summary["percentage_within_malignant_patients"].sum(), 3),
]

patient_malignancy_summary

,n_malignant_lesions,count,percentage,cumulative_percentage,percentage_within_malignant_patients
0,0.0,783,75.144,75.144,NaN
1,1.0,193,18.522,93.666,74.517
2,2.0,35,3.359,97.025,13.514
3,3.0,18,1.727,98.752,6.950
4,4.0,6,0.576,99.328,2.317
5,5.0,2,0.192,99.52,0.772
6,6.0,1,0.096,99.616,0.386
7,7.0,2,0.192,99.808,0.772
8,8.0,1,0.096,99.904,0.386
9,14.0,1,0.096,100.0,0.386


In [68]:
malignant_per_patient.describe()

count    1042.000000
mean        0.377159
std         0.922957
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        14.000000
Name: malignant, dtype: float64

In [77]:
# n_strong_labels (df2) per patient

df_patient_dx = df3[["isic_id", "patient_id"]].merge(
    df2[["isic_id", "iddx_1", "iddx_2"]],
    on="isic_id",
    how="inner",
    validate="one_to_one",
)

# Define strong label
df_patient_dx["is_specific"] = df_patient_dx["iddx_2"].notna()

df_patient_dx["benign_specific"] = (
    df_patient_dx["iddx_1"] == "Benign"
) & df_patient_dx["is_specific"]

df_patient_dx["indeterminate_specific"] = (
    df_patient_dx["iddx_1"] == "Indeterminate"
) & df_patient_dx["is_specific"]

df_patient_dx["malignant_specific"] = (
    df_patient_dx["iddx_1"] == "Malignant"
) & df_patient_dx["is_specific"]

df_patient_dx["benign_not_specific"] = (df_patient_dx["iddx_1"] == "Benign") & (
    ~df_patient_dx["is_specific"]
)

# Patient-level table
patient_specific_summary = (
    df_patient_dx.groupby("patient_id")
    .agg(
        n_lesions=("isic_id", "count"),
        n_benign_specific=("benign_specific", "sum"),
        n_indeterminate_specific=("indeterminate_specific", "sum"),
        n_malignant_specific=("malignant_specific", "sum"),
        n_benign_not_specific=("benign_not_specific", "sum"),
    )
    .reset_index()
)

# Percentages over total lesions per patient
for col in [
    "n_benign_specific",
    "n_indeterminate_specific",
    "n_malignant_specific",
    "n_benign_not_specific",
]:
    patient_specific_summary[col.replace("n_", "pct_")] = round(
        100 * patient_specific_summary[col] / patient_specific_summary["n_lesions"], 3
    )

patient_specific_summary_all = patient_specific_summary.copy()

patient_specific_summary_specific = patient_specific_summary_all[
    (
        patient_specific_summary_all["n_benign_specific"]
        + patient_specific_summary_all["n_indeterminate_specific"]
        + patient_specific_summary_all["n_malignant_specific"]
    )
    > 0
].copy()

patient_specific_summary_specific

,patient_id,n_lesions,n_benign_specific,n_indeterminate_specific,n_malignant_specific,n_benign_not_specific,pct_benigpct_specific,pct_indeterminate_specific,pct_malignant_specific,pct_benigpct_not_specific
1,IP_0014998,352,1,0,0,351,0.284,0.0,0.000,99.716
2,IP_0023586,939,0,0,1,938,0.000,0.0,0.106,99.894
3,IP_0028775,212,1,0,0,211,0.472,0.0,0.000,99.528
4,IP_0028993,225,1,0,0,224,0.444,0.0,0.000,99.556
6,IP_0039318,132,2,0,0,130,1.515,0.0,0.000,98.485
...,...,...,...,...,...,...,...,...,...,...
1034,IP_9890051,303,1,0,0,302,0.330,0.0,0.000,99.670
1037,IP_9916724,221,2,0,0,219,0.905,0.0,0.000,99.095
1038,IP_9970422,187,1,0,0,186,0.535,0.0,0.000,99.465
1039,IP_9978624,151,0,0,1,150,0.000,0.0,0.662,99.338


In [78]:
def get_specific_pattern(row):
    has_benign = row["n_benign_specific"] > 0
    has_indeterminate = row["n_indeterminate_specific"] > 0
    has_malignant = row["n_malignant_specific"] > 0

    if has_benign and not has_indeterminate and not has_malignant:
        return "only benign specific"

    if has_malignant and not has_benign and not has_indeterminate:
        return "only malignant specific"

    if has_indeterminate and not has_benign and not has_malignant:
        return "only indeterminate specific"

    if has_benign and has_indeterminate and not has_malignant:
        return "benign + indeterminate specific"

    if has_benign and has_malignant and not has_indeterminate:
        return "benign + malignant specific"

    if has_indeterminate and has_malignant and not has_benign:
        return "indeterminate + malignant specific"

    if has_benign and has_indeterminate and has_malignant:
        return "all 3 specific"

    return "other"


patient_specific_summary_specific["specific_diagnosis_pattern"] = (
    patient_specific_summary_specific.apply(
        get_specific_pattern,
        axis=1,
    )
)

n_all_patients = df3["patient_id"].nunique()
n_patients_with_specific_dx = len(patient_specific_summary_specific)

patient_specific_pattern_summary = (
    patient_specific_summary_specific["specific_diagnosis_pattern"]
    .value_counts()
    .rename_axis("specific_diagnosis_pattern")
    .reset_index(name="count")
)

patient_specific_pattern_summary["percentage_all_patients"] = round(
    100 * patient_specific_pattern_summary["count"] / n_all_patients,
    3,
)

patient_specific_pattern_summary["percentage_patients_with_specific_dx"] = round(
    100 * patient_specific_pattern_summary["count"] / n_patients_with_specific_dx,
    3,
)

patient_specific_pattern_summary.loc[len(patient_specific_pattern_summary)] = [
    "Total",
    patient_specific_pattern_summary["count"].sum(),
    round(
        patient_specific_pattern_summary["percentage_all_patients"].sum(),
        3,
    ),
    round(
        patient_specific_pattern_summary["percentage_patients_with_specific_dx"].sum(),
        3,
    ),
]

patient_specific_pattern_summary

,specific_diagnosis_pattern,count,percentage_all_patients,percentage_patients_with_specific_dx
0,only benign specific,336,32.246,50.909
1,only malignant specific,185,17.754,28.030
2,benign + malignant specific,47,4.511,7.121
3,only indeterminate specific,42,4.031,6.364
4,benign + indeterminate specific,23,2.207,3.485
5,indeterminate + malignant specific,15,1.440,2.273
6,all 3 specific,12,1.152,1.818
7,Total,660,63.341,100.000


In [79]:
patient_specific_summary_specific[
    [
        "n_benign_specific",
        "n_indeterminate_specific",
        "n_malignant_specific",
        "n_benign_not_specific",
    ]
].describe()

,n_benign_specific,n_indeterminate_specific,n_malignant_specific,n_benign_not_specific
count,660.000000,660.000000,660.000000,660.000000
mean,0.850000,0.172727,0.595455,466.995455
std,0.853583,0.487023,1.102478,619.309203
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,175.000000
50%,1.000000,0.000000,0.000000,312.000000
75%,1.000000,0.000000,1.000000,581.250000
max,5.000000,4.000000,14.000000,9182.000000


In [80]:
# age_approx

age_na_summary = pd.DataFrame(
    {
        "count": [
            df3["age_approx"].isna().sum(),
            df3["age_approx"].notna().sum(),
            len(df3),
        ]
    },
    index=["NA", "Non-NA", "Total"],
)

age_na_summary["percentage"] = round(100 * age_na_summary["count"] / len(df3), 2)

age_na_summary

,count,percentage
NA,2798,0.7
Non-NA,398261,99.3
Total,401059,100.0


In [81]:
age_by_patient = df3.groupby("patient_id", dropna=False)["age_approx"].first()

age_patient_na_summary = pd.DataFrame(
    {
        "count": [
            age_by_patient.isna().sum(),
            age_by_patient.notna().sum(),
            len(age_by_patient),
        ]
    },
    index=["NA", "Non-NA", "Total"],
)

age_patient_na_summary["percentage"] = round(
    100 * age_patient_na_summary["count"] / len(age_by_patient), 2
)

age_patient_na_summary

,count,percentage
NA,13,1.25
Non-NA,1029,98.75
Total,1042,100.00


In [119]:
# Patients with missing age

patients_missing_age = (
    df3.loc[df3["age_approx"].isna(), "patient_id"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
    .to_list()
)

patients_missing_age

['IP_0212694',
 'IP_0362830',
 'IP_1076642',
 'IP_1129512',
 'IP_3371534',
 'IP_3374234',
 'IP_3647018',
 'IP_6659956',
 'IP_6865940',
 'IP_7013759',
 'IP_7848492',
 'IP_7858959',
 'IP_9652562']

In [86]:
# Subset containing all rows with missing age
df_age_na = df3[df3["age_approx"].isna()].copy()

n_rows_na = len(df_age_na)
n_patients_na = df_age_na["patient_id"].nunique()

print(f"Rows with missing age: {n_rows_na}")
print(f"Patients with missing age: {n_patients_na}")

# Patients with at least one missing age value
patients_na = set(df_age_na["patient_id"])

# All rows belonging to those patients
all_rows_patients_na = df3[df3["patient_id"].isin(patients_na)]

print("Rows belonging to the 13 patients:", len(all_rows_patients_na))
print("Rows with missing age:", len(df_age_na))

print("Perfect correspondence:", len(all_rows_patients_na) == len(df_age_na))

Rows with missing age: 2798
Patients with missing age: 13
Rows belonging to the 13 patients: 2798
Rows with missing age: 2798
Perfect correspondence: True


In [87]:
# Merge attribution information
df_age_na_attr = df3[["isic_id", "patient_id"]].merge(
    df2[["isic_id", "attribution"]], on="isic_id", how="inner", validate="one_to_one"
)

hospital_summary = (
    df_age_na_attr[df_age_na_attr["patient_id"].isin(patients_na)]
    .groupby("attribution")
    .agg(n_patients=("patient_id", "nunique"), n_images=("patient_id", "size"))
    .sort_values("n_images", ascending=False)
)

hospital_summary

,n_patients,n_images
attribution,,
"Department of Dermatology, Hospital Clínic de Barcelona",4,1461
"Frazer Institute, The University of Queensland, Dermatology Research Centre",5,1205
University Hospital of Basel,4,132


In [99]:
# Merge diagnostic information
df_age_na_dx = df3[df3["patient_id"].isin(patients_na)][
    ["isic_id", "patient_id"]
].merge(
    df2[["isic_id", "iddx_1", "iddx_2"]],
    on="isic_id",
    how="inner",
    validate="one_to_one",
)

patient_summary = df_age_na_dx.groupby("patient_id").agg(
    total_lesions=("isic_id", "size"),
    benign_lesions=("iddx_1", lambda x: (x == "Benign").sum()),
    indeterminate_lesions=("iddx_1", lambda x: (x == "Indeterminate").sum()),
    malignant_lesions=("iddx_1", lambda x: (x == "Malignant").sum()),
    specific_lesions=("iddx_2", lambda x: x.notna().sum()),
    non_specific_lesions=("iddx_2", lambda x: x.isna().sum()),
)

for column in [
    "benign_lesions",
    "indeterminate_lesions",
    "malignant_lesions",
    "specific_lesions",
    "non_specific_lesions",
]:
    patient_summary[f"{column}_pct"] = round(
        100 * patient_summary[column] / patient_summary["total_lesions"], 2
    )

patient_summary

,total_lesions,benign_lesions,indeterminate_lesions,malignant_lesions,specific_lesions,non_specific_lesions,benign_lesions_pct,indeterminate_lesions_pct,malignant_lesions_pct,specific_lesions_pct,non_specific_lesions_pct
patient_id,,,,,,,,,,,
IP_0212694,319,319,0,0,1,318,100.00,0.00,0.00,0.31,99.69
IP_0362830,93,93,0,0,0,93,100.00,0.00,0.00,0.00,100.00
IP_1076642,47,47,0,0,0,47,100.00,0.00,0.00,0.00,100.00
IP_1129512,53,53,0,0,0,53,100.00,0.00,0.00,0.00,100.00
IP_3371534,903,902,0,1,1,902,99.89,0.00,0.11,0.11,99.89
IP_3374234,7,7,0,0,0,7,100.00,0.00,0.00,0.00,100.00
IP_3647018,86,85,0,1,1,85,98.84,0.00,1.16,1.16,98.84
IP_6659956,13,13,0,0,0,13,100.00,0.00,0.00,0.00,100.00
IP_6865940,24,24,0,0,0,24,100.00,0.00,0.00,0.00,100.00


In [105]:
# Number of unique non-missing age values per patient

age_consistency = (
    df3.groupby("patient_id")["age_approx"]
    .nunique(dropna=True)
    .value_counts()
    .sort_index()
    .rename_axis("n_unique_age_values")
    .reset_index(name="n_patients")
)

age_consistency

,n_unique_age_values,n_patients
0,0,13
1,1,929
2,2,100


In [122]:
# Patients with more than one age value

patients_multiple_ages = (
    df3.groupby("patient_id")["age_approx"]
    .nunique(dropna=True)
    .loc[lambda x: x > 1]
    .sort_values(ascending=False)
)

age_values_per_patient = (
    df3[df3["patient_id"].isin(patients_multiple_ages.index)]
    .groupby("patient_id")["age_approx"]
    .unique()
    .reset_index()
)

age_values_per_patient

,patient_id,age_approx
0,IP_0159369,"[70.0, 65.0]"
1,IP_0166385,"[55.0, 60.0]"
2,IP_0199195,"[65.0, 60.0]"
3,IP_0826114,"[55.0, 50.0]"
4,IP_0878241,"[70.0, 65.0]"
...,...,...
95,IP_9553357,"[55.0, 50.0]"
96,IP_9587089,"[50.0, 45.0]"
97,IP_9588095,"[45.0, 40.0]"
98,IP_9807499,"[40.0, 35.0]"


In [125]:
# Age range among patients with multiple recorded ages

age_range_per_patient = df3.groupby("patient_id")["age_approx"].agg(
    min_age="min", max_age="max", n_unique_ages=lambda x: x.nunique(dropna=True)
)

age_range_per_patient["age_difference"] = (
    age_range_per_patient["max_age"] - age_range_per_patient["min_age"]
)

age_range_per_patient = age_range_per_patient[
    age_range_per_patient["n_unique_ages"] > 1
]

age_range_per_patient.head()

,min_age,max_age,n_unique_ages,age_difference
patient_id,,,,
IP_0159369,65.0,70.0,2,5.0
IP_0166385,55.0,60.0,2,5.0
IP_0199195,60.0,65.0,2,5.0
IP_0826114,50.0,55.0,2,5.0
IP_0878241,65.0,70.0,2,5.0


In [129]:
age_range_per_patient["age_difference"].value_counts().sort_index()

age_difference
5.0    100
Name: count, dtype: int64

In [130]:
# Verify that all patients with multiple ages follow a 5-year increment pattern

age_values_per_patient = df3.groupby("patient_id")["age_approx"].apply(
    lambda x: sorted(x.dropna().unique())
)

all_age_transitions_are_5y = (
    age_values_per_patient.loc[age_values_per_patient.apply(len) > 1]
    .apply(lambda x: (len(x) == 2 and x[1] - x[0] == 5))
    .all()
)

print(all_age_transitions_are_5y)

True


In [128]:
age_values_per_patient = df3.groupby("patient_id")["age_approx"].apply(
    lambda x: sorted(x.dropna().unique())
)

age_transition_summary = (
    age_values_per_patient.loc[age_values_per_patient.apply(len) > 1]
    .apply(lambda x: f"{x[0]}-{x[1]}")
    .value_counts()
    .sort_index()
    .rename_axis("age_transition")
    .reset_index(name="n_patients")
)

age_transition_summary

,age_transition,n_patients
0,20.0-25.0,3
1,25.0-30.0,1
2,30.0-35.0,1
3,35.0-40.0,4
4,40.0-45.0,10
5,45.0-50.0,12
6,50.0-55.0,13
7,55.0-60.0,19
8,60.0-65.0,16
9,65.0-70.0,9


In [135]:
# Patient-level adjusted age:
# - If a patient has one age: keep that age
# - If a patient has two ages differing by 5 years: use the midpoint
# - If all ages are missing: keep NA

patient_age_adjusted = (
    df3.groupby("patient_id")["age_approx"]
    .apply(lambda x: sorted(x.dropna().unique()))
    .reset_index(name="age_values")
)

patient_age_adjusted["n_unique_ages"] = patient_age_adjusted["age_values"].apply(len)

patient_age_adjusted["age_adjusted"] = patient_age_adjusted["age_values"].apply(
    lambda x: np.nan if len(x) == 0 else np.mean(x)
)

patient_age_adjusted[["patient_id", "age_adjusted"]]

,patient_id,age_adjusted
0,IP_0008821,65.0
1,IP_0014998,65.0
2,IP_0023586,75.0
3,IP_0028775,50.0
4,IP_0028993,30.0
...,...,...
1037,IP_9916724,35.0
1038,IP_9970422,75.0
1039,IP_9978624,55.0
1040,IP_9979014,20.0


In [136]:
# Age summary
patient_age_adjusted.describe()

,n_unique_ages,age_adjusted
count,1042.000000,1029.000000
mean,1.083493,52.609329
std,0.318703,14.916889
min,0.000000,5.000000
25%,1.000000,40.000000
50%,1.000000,55.000000
75%,1.000000,65.000000
max,2.000000,85.000000


In [137]:
# Add adjusted patient age back to lesion-level dataframe

df3_age_adjusted = df3.merge(
    patient_age_adjusted[["patient_id", "age_adjusted"]],
    on="patient_id",
    how="left",
    validate="many_to_one",
)

# Age distribution using adjusted age

age_adjusted_distribution = (
    df3_age_adjusted.assign(age=df3_age_adjusted["age_adjusted"].fillna("NA"))
    .groupby("age")
    .agg(
        n_lesions_total=("isic_id", "size"), n_patients_total=("patient_id", "nunique")
    )
    .reset_index()
)

age_adjusted_distribution["perc_lesions_total"] = round(
    100
    * age_adjusted_distribution["n_lesions_total"]
    / age_adjusted_distribution["n_lesions_total"].sum(),
    2,
)

age_adjusted_distribution["perc_patients_total"] = round(
    100
    * age_adjusted_distribution["n_patients_total"]
    / age_adjusted_distribution["n_patients_total"].sum(),
    2,
)

age_adjusted_distribution["sort_age"] = pd.to_numeric(
    age_adjusted_distribution["age"], errors="coerce"
)

age_adjusted_distribution = (
    age_adjusted_distribution.sort_values("sort_age", na_position="last")
    .drop(columns="sort_age")
    .reset_index(drop=True)
)

age_adjusted_distribution = age_adjusted_distribution[
    [
        "age",
        "n_lesions_total",
        "perc_lesions_total",
        "n_patients_total",
        "perc_patients_total",
    ]
]

age_adjusted_distribution

,age,n_lesions_total,perc_lesions_total,n_patients_total,perc_patients_total
0,5.0,1,0.00,1,0.10
1,15.0,644,0.16,7,0.67
2,20.0,1739,0.43,16,1.54
3,22.5,351,0.09,3,0.29
4,25.0,3084,0.77,25,2.40
5,27.5,295,0.07,1,0.10
6,30.0,10104,2.52,55,5.28
7,32.5,183,0.05,1,0.10
8,35.0,11357,2.83,58,5.57
9,37.5,1325,0.33,4,0.38


In [ ]:
# Sex

# Number of unique non-missing sex values per patient

sex_consistency = (
    df3.groupby("patient_id")["sex"]
    .nunique(dropna=True)
    .value_counts()
    .sort_index()
    .rename_axis("n_unique_sex_values")
    .reset_index(name="n_patients")
)

sex_consistency

,n_unique_sex_values,n_patients
0,0,33
1,1,1009


In [117]:
# Patients with missing sex

patients_missing_sex = (
    df3.loc[df3["sex"].isna(), "patient_id"].drop_duplicates().sort_values().to_list()
)

patients_missing_sex

['IP_0473747',
 'IP_0797650',
 'IP_1016097',
 'IP_1241473',
 'IP_1433033',
 'IP_1474479',
 'IP_1715157',
 'IP_1822913',
 'IP_2075317',
 'IP_2148283',
 'IP_2914511',
 'IP_3371534',
 'IP_4080836',
 'IP_4579352',
 'IP_5044083',
 'IP_5373245',
 'IP_5887935',
 'IP_6187331',
 'IP_6193088',
 'IP_6207479',
 'IP_6413837',
 'IP_6709102',
 'IP_6732258',
 'IP_6975913',
 'IP_7563441',
 'IP_7703522',
 'IP_7858959',
 'IP_8036435',
 'IP_8377208',
 'IP_8810058',
 'IP_8999199',
 'IP_9472195',
 'IP_9550509']

In [ ]:
# Missing values at lesion level

sex_lesion_summary = pd.DataFrame(
    {"count": [df3["sex"].isna().sum(), df3["sex"].notna().sum(), len(df3)]},
    index=["NA", "Non-NA", "Total"],
)

sex_lesion_summary["percentage"] = round(
    100 * sex_lesion_summary["count"] / len(df3), 2
)

sex_lesion_summary

,count,percentage
NA,11517,2.87
Non-NA,389542,97.13
Total,401059,100.00


In [101]:
# One sex value per patient
sex_by_patient = df3.groupby("patient_id", dropna=False)["sex"].first()

sex_patient_summary = pd.DataFrame(
    {
        "count": [
            sex_by_patient.isna().sum(),
            sex_by_patient.notna().sum(),
            len(sex_by_patient),
        ]
    },
    index=["NA", "Non-NA", "Total"],
)

sex_patient_summary["percentage"] = round(
    100 * sex_patient_summary["count"] / len(sex_by_patient), 2
)

sex_patient_summary

,count,percentage
NA,33,3.17
Non-NA,1009,96.83
Total,1042,100.00


In [102]:
# Rows with missing sex
df_sex_na = df3[df3["sex"].isna()].copy()

patients_sex_na = set(df_sex_na["patient_id"])

# All rows belonging to those patients
all_rows_patients_sex_na = df3[df3["patient_id"].isin(patients_sex_na)]

print("Rows with missing sex:", len(df_sex_na))
print("Rows belonging to affected patients:", len(all_rows_patients_sex_na))

print("Perfect correspondence:", len(df_sex_na) == len(all_rows_patients_sex_na))

Rows with missing sex: 11517
Rows belonging to affected patients: 11517
Perfect correspondence: True


In [104]:
# Combined lesion-level and patient-level summary

sex_lesion_distribution = (
    df3["sex"]
    .fillna("NA")
    .value_counts()
    .rename_axis("sex")
    .reset_index(name="n_lesions_total")
)

sex_patient_distribution = (
    df3.groupby("patient_id")["sex"]
    .first()
    .fillna("NA")
    .value_counts()
    .rename_axis("sex")
    .reset_index(name="n_patients_total")
)

sex_summary = sex_lesion_distribution.merge(
    sex_patient_distribution, on="sex", how="outer", validate="one_to_one"
)

sex_summary["perc_lesions_total"] = round(
    100 * sex_summary["n_lesions_total"] / sex_summary["n_lesions_total"].sum(), 2
)

sex_summary["perc_patients_total"] = round(
    100 * sex_summary["n_patients_total"] / sex_summary["n_patients_total"].sum(), 2
)

sex_summary = sex_summary[
    [
        "sex",
        "n_lesions_total",
        "perc_lesions_total",
        "n_patients_total",
        "perc_patients_total",
    ]
]

sex_summary

,sex,n_lesions_total,perc_lesions_total,n_patients_total,perc_patients_total
0,NA,11517,2.87,33,3.17
1,female,123996,30.92,458,43.95
2,male,265546,66.21,551,52.88


In [110]:
# Patients with missing sex

patients_sex_na = df3.loc[df3["sex"].isna(), "patient_id"].unique()

n_patients_sex_na = len(patients_sex_na)

print(f"Patients with missing sex: {n_patients_sex_na}")

# Age availability among patients with missing sex

patient_demographics = df3.groupby("patient_id").agg(
    age_approx=("age_approx", "first"), sex=("sex", "first")
)

sex_na_patients = patient_demographics.loc[patient_demographics["sex"].isna()]

n_age_na = sex_na_patients["age_approx"].isna().sum()
n_age_non_na = sex_na_patients["age_approx"].notna().sum()

age_availability_summary = pd.DataFrame(
    {"count": [n_age_na, n_age_non_na, len(sex_na_patients)]},
    index=["Age missing", "Age available", "Total"],
)

age_availability_summary["percentage"] = round(
    100 * age_availability_summary["count"] / len(sex_na_patients), 2
)

age_availability_summary

Patients with missing sex: 33


,count,percentage
Age missing,2,6.06
Age available,31,93.94
Total,33,100.00


In [111]:
# Attribution of patients with missing sex

sex_na_hospitals = df3[df3["patient_id"].isin(patients_sex_na)][
    ["isic_id", "patient_id"]
].merge(
    df2[["isic_id", "attribution"]], on="isic_id", how="inner", validate="one_to_one"
)

hospital_summary = (
    sex_na_hospitals.groupby("attribution")
    .agg(n_patients=("patient_id", "nunique"), n_lesions=("isic_id", "size"))
    .sort_values("n_lesions", ascending=False)
)

hospital_summary

,n_patients,n_lesions
attribution,,
University Hospital of Basel,27,8744
"Frazer Institute, The University of Queensland, Dermatology Research Centre",5,2319
"Department of Dermatology, Hospital Clínic de Barcelona",1,454


In [112]:
# Merge diagnostic information

df_sex_na_dx = df3[df3["patient_id"].isin(patients_sex_na)][
    ["isic_id", "patient_id"]
].merge(
    df2[["isic_id", "iddx_1", "iddx_2"]],
    on="isic_id",
    how="inner",
    validate="one_to_one",
)

total_lesions = len(df_sex_na_dx)

cohort_summary = pd.DataFrame(
    {
        "count": [
            (df_sex_na_dx["iddx_1"] == "Benign").sum(),
            (df_sex_na_dx["iddx_1"] == "Indeterminate").sum(),
            (df_sex_na_dx["iddx_1"] == "Malignant").sum(),
            (
                (df_sex_na_dx["iddx_1"] == "Benign") & df_sex_na_dx["iddx_2"].notna()
            ).sum(),
            (
                (df_sex_na_dx["iddx_1"] == "Indeterminate")
                & df_sex_na_dx["iddx_2"].notna()
            ).sum(),
            (
                (df_sex_na_dx["iddx_1"] == "Malignant") & df_sex_na_dx["iddx_2"].notna()
            ).sum(),
            df_sex_na_dx["iddx_2"].isna().sum(),
            total_lesions,
        ]
    },
    index=[
        "Benign lesions",
        "Indeterminate lesions",
        "Malignant lesions",
        "Benign specific lesions",
        "Indeterminate specific lesions",
        "Malignant specific lesions",
        "Non-specific lesions",
        "Total lesions",
    ],
)

cohort_summary["percentage"] = round(100 * cohort_summary["count"] / total_lesions, 2)

cohort_summary

,count,percentage
Benign lesions,11502,99.87
Indeterminate lesions,5,0.04
Malignant lesions,10,0.09
Benign specific lesions,17,0.15
Indeterminate specific lesions,5,0.04
Malignant specific lesions,10,0.09
Non-specific lesions,11485,99.72
Total lesions,11517,100.00


In [113]:
# Patient-level diagnostic summary for patients with missing sex

patient_summary_sex_na = df_sex_na_dx.groupby("patient_id").agg(
    total_lesions=("isic_id", "size"),
    benign_lesions=("iddx_1", lambda x: (x == "Benign").sum()),
    indeterminate_lesions=("iddx_1", lambda x: (x == "Indeterminate").sum()),
    malignant_lesions=("iddx_1", lambda x: (x == "Malignant").sum()),
    specific_lesions=("iddx_2", lambda x: x.notna().sum()),
    non_specific_lesions=("iddx_2", lambda x: x.isna().sum()),
)

for column in [
    "benign_lesions",
    "indeterminate_lesions",
    "malignant_lesions",
    "specific_lesions",
    "non_specific_lesions",
]:
    patient_summary_sex_na[f"{column}_pct"] = round(
        100 * patient_summary_sex_na[column] / patient_summary_sex_na["total_lesions"],
        2,
    )

patient_summary_sex_na

,total_lesions,benign_lesions,indeterminate_lesions,malignant_lesions,specific_lesions,non_specific_lesions,benign_lesions_pct,indeterminate_lesions_pct,malignant_lesions_pct,specific_lesions_pct,non_specific_lesions_pct
patient_id,,,,,,,,,,,
IP_0473747,262,262,0,0,1,261,100.00,0.00,0.00,0.38,99.62
IP_0797650,329,329,0,0,0,329,100.00,0.00,0.00,0.00,100.00
IP_1016097,454,453,0,1,1,453,99.78,0.00,0.22,0.22,99.78
IP_1241473,251,250,0,1,1,250,99.60,0.00,0.40,0.40,99.60
IP_1433033,313,313,0,0,0,313,100.00,0.00,0.00,0.00,100.00
IP_1474479,1084,1083,0,1,3,1081,99.91,0.00,0.09,0.28,99.72
IP_1715157,223,218,2,3,6,217,97.76,0.90,1.35,2.69,97.31
IP_1822913,153,153,0,0,0,153,100.00,0.00,0.00,0.00,100.00
IP_2075317,54,54,0,0,0,54,100.00,0.00,0.00,0.00,100.00


In [ ]:
# anatomy

## metadata (df3) summary

Image-level metadata and automatically extracted lesion features.

- **Rows**: 401059. No duplicated rows.
- **Cols**: 42
- **Col names**: [isic_id, patient_id, age_approx, sex, anatom_site_general, clin_size_long_diam_mm, image_type, tbp_tile_type, tbp_lv_A, tbp_lv_Aext, tbp_lv_B, tbp_lv_Bext, tbp_lv_C, tbp_lv_Cext, tbp_lv_H, tbp_lv_Hext, tbp_lv_L, tbp_lv_Lext, tbp_lv_areaMM2, tbp_lv_area_perim_ratio, tbp_lv_color_std_mean, tbp_lv_deltaA, tbp_lv_deltaB, tbp_lv_deltaL, tbp_lv_deltaLB, tbp_lv_deltaLBnorm, tbp_lv_eccentricity, tbp_lv_location, tbp_lv_location_simple, tbp_lv_minorAxisMM, tbp_lv_nevi_confidence, tbp_lv_norm_border, tbp_lv_norm_color, tbp_lv_perimeterMM, tbp_lv_radial_color_std_max, tbp_lv_stdL, tbp_lv_stdLExt, tbp_lv_symm_2axis, tbp_lv_symm_2axis_angle, tbp_lv_x, tbp_lv_y, tbp_lv_z]

Given the large number of metadata columns, variables will be grouped before detailed analysis:

1. **Identifiers**: `isic_id`, `patient_id`.
    - Used to track lesions and patients.
    
    - ***isic_id***: unique id per lesion (pkey). 0 NA. 100% match with df1.

    - ***patient_id***: patient id. 0 NA. 1,042 unique patients and 401,059 lesions, corresponding to a median of 241.5 and mean of 384.9 lesions per patient (IQR: 115.0 – 477.5). The number of lesions per patient is highly heterogeneous, ranging from 1 to 9,184 lesions, indicating that observations are strongly clustered within patients.

        When linked to the ground-truth labels (df1), 783 patients (75.1%) had no malignant lesions, whereas 259 patients (24.9%) had at least one malignant lesion. Among patients with malignant lesions, most had only a small number of malignant lesions:

        | Number of malignant lesions | Patients (n) | % of all patients | % of patients with ≥1 malignant lesion |
        | --------------------------- | -----------: | ----------------: | -------------------------------------: |
        | 1                           |          193 |              18.5 |                                   74.5 |
        | 2                           |           35 |               3.4 |                                   13.5 |
        | 3                           |           18 |               1.7 |                                    6.9 |
        | 4                           |            6 |             < 0.6 |                                    2.3 |
        | 5                           |            2 |             < 0.2 |                                  < 0.8 |
        | 6                           |            1 |             < 0.1 |                                  < 0.4 |
        | 7                           |            2 |             < 0.2 |                                  < 0.8 |
        | 8                           |            1 |             < 0.1 |                                  < 0.4 |
        | 14                          |            1 |             < 0.1 |                                  < 0.4 |


        These results suggest that malignant lesions are concentrated in a relatively small subset of patients. Furthermore, because multiple lesions belong to the same patient, observations cannot be considered independent. Therefore, **patient-level train/test splitting should be used in downstream modeling to avoid data leakage**.

        When linked to the iddx_* labels (df2), 660 patients (63.3%) had at least one lesion with a specific diagnosis (`iddx_2` non-null), whereas the remaining 382 patients (36.7%) only contributed weakly-labelled benign lesions. Among patients with at least one specific diagnosis, the distribution of diagnostic patterns was:

        | Pattern                           | Patients (n) | All patients (%) | Within specific-diagnosis patients (%) |
        |-----------------------------------|--------------|------------------|-----------------------------------------|
        | Only benign                       |          336 |             32.2 |                                    50.9 |
        | Only malignant                    |          185 |             17.8 |                                    28.0 |
        | Only indeterminate                |           42 |              4.0 |                                     6.4 |
        | Benign + malignant                |           47 |              4.5 |                                     7.1 |
        | Benign + indeterminate            |           23 |              2.2 |                                     3.5 |
        | Indeterminate + malignant         |           15 |              1.4 |                                     2.3 |
        | Benign + indeterminate + malignant|           12 |              1.2 |                                     1.8 |
        | **Total**                         |      **660** |         **63.3** |                               **100.0** |

        Patients with at least one specific diagnosis typically contributed a very small number of biopsied / specifically diagnosed lesions together with a large number of weakly-labelled benign lesions. The median patient had:

        - 1 benign-specific lesion (IQR: 0–1),
        - 0 indeterminate-specific lesions (IQR: 0–0),
        - 0 malignant-specific lesions (IQR: 0–1),
        - 312 benign non-specific lesions (IQR: 175–581).

2. **Clinical metadata**: `age_approx`, `sex`, `anatom_site_general`.
    
    - ***age_approx***: approximate patient age at imaging. There are 2,798 (0.7%) missing values at lesion level. At patient level, missing age values correspond to 13 patients, representing 1.25% of the 1,042 patients.

    All 2,798 missing-age rows belong exactly to these 13 patients, indicating complete absence of age information rather than sporadic missing values. These patients come from three institutions: 
        - Hospital Clínic de Barcelona (4 patients, 1,461 images).
        - Frazer Institute / The University of Queensland (5 patients, 1,205 images).
        - University Hospital of Basel (4 patients, 132 images).

    The 13 patients with missing age contribute mostly non-specific benign lesions. In total, they contribute 2,798 lesions: 2,794 benign lesions (5 of which are specific), 1 indeterminate lesion and 3 malignant lesions. So only 8 lesions have a specific diagnosis (`iddx_2` non-null), while 2,790 lesions are non-specific. Therefore, missing age appears to affect a small subset of patients and is mostly associated with weakly-labelled benign observations.

    These 13 patients are: ['IP_0212694', 'IP_0362830', 'IP_1076642', 'IP_1129512', 'IP_3371534', 'IP_3374234', 'IP_3647018', 'IP_6659956', 'IP_6865940', 'IP_7013759', 'IP_7848492', 'IP_7858959', 'IP_9652562'].

    Age values are strongly discretized, mostly in 5-year intervals. Out of 1042 patients:
        - 13 have missing age.
        - 929 have a single recorded age value. 
        - However, 100 patients have two recorded age values, always differing by exactly 5 years (for example 50 and 55 or 60 and 65). This suggests that these patients contributed images acquired across different time points or visits. For patient-level summaries, these patients were assigned the midpoint of their age interval. This avoids counting the same patient in two different age groups.

    After this adjustment, patient-level age ranges from 5 to 85 years, with a mean of 52.6 years, a median of 55 years, and an interquartile range of 40–65 years, indicating that most lesions come from middle-aged and older patients.

        | Age | n_lesions_total | perc_lesions_total | n_patients_total | perc_patients_total |
        |-----|----------------:|-------------------:|-----------------:|--------------------:|
        | 5   |               1 |               0.00 |                1 |                0.10 |
        | 15  |             644 |               0.16 |                7 |                0.67 |
        | 20  |            1742 |               0.43 |               16 |                1.54 |
        | 25  |            3433 |               0.86 |               28 |                2.69 |
        | 30  |           10400 |               2.59 |               56 |                5.37 |
        | 35  |           11543 |               2.88 |               59 |                5.66 |
        | 40  |           31297 |               7.80 |              107 |               10.27 |
        | 45  |           23580 |               5.88 |               85 |                8.16 |
        | 50  |           47924 |              11.95 |              119 |               11.42 |
        | 55  |           58123 |              14.49 |              133 |               12.76 |
        | 60  |           54109 |              13.49 |              122 |               11.71 |
        | 65  |           54946 |              13.70 |              129 |               12.38 |
        | 70  |           39775 |               9.92 |               76 |                7.29 |
        | 75  |           30801 |               7.68 |               46 |                4.41 |
        | 80  |           21096 |               5.26 |               33 |                3.17 |
        | 85  |            8847 |               2.21 |               12 |                1.15 |
        | NA  |            2798 |               0.70 |               13 |                1.25 |
 
    - ***sex***: patient sex: 'female' and 'male' or 'NA'. Each patient has a single sex record. 

    A total of 33 patients have missing sex information, corresponding to a small proportion of both patients and images. These patients come from three institutions: 
        - Hospital Clínic de Barcelona (1 patients, 454 images).
        - Frazer Institute / The University of Queensland (5 patients, 2,319 images).
        - University Hospital of Basel (27 patients, 8,744 images). 
    
    These 33 patients are: ['IP_0473747', 'IP_0797650', 'IP_1016097', 'IP_1241473', 'IP_1433033', 'IP_1474479', 'IP_1715157', 'IP_1822913', 'IP_2075317', 'IP_2148283', 'IP_2914511', 'IP_3371534', 'IP_4080836', 'IP_4579352', 'IP_5044083', 'IP_5373245', 'IP_5887935', 'IP_6187331', 'IP_6193088', 'IP_6207479', 'IP_6413837', 'IP_6709102', 'IP_6732258', 'IP_6975913', 'IP_7563441', 'IP_7703522', 'IP_7858959', 'IP_8036435', 'IP_8377208', 'IP_8810058', 'IP_8999199', 'IP_9472195', 'IP_9550509']

    Among these 33 patients, only 2 also have missing age information ('IP_3371534' and 'IP_7858959'), whereas age is available for the remaining 31 patients. Therefore, overlap between missing sex and missing age is limited.

    These 33 patients (3.17%) contribute 11,517 lesions in total (2.88%). Diagnostic review shows that 11,502 lesions (99.87%) are classified as benign, 5 (0.04%) as indeterminate, and 10 (0.09%) as malignant. Only 32 lesions (0.28%) have a specific diagnosis (iddx_2 non-null). Among these, 17 are benign, 5 are indeterminate, and 10 are malignant. 

    Among the 1,009 patients with available sex information, 551 (54.61%) are male and 458 (45.39%) are female. At lesion level, 265,546 images (68.17%) correspond to male patients and 123,996 images (31.83%) to female patients. The larger difference observed at lesion level suggests that male patients contribute more images on average than female patients.
    
    - ***sex***:

3. **Technical acquisition variables**: `image_type`, `tbp_tile_type`.
    - Useful for dataset description and possible bias assessment.
    - Probably not used as predictors unless explicitly modeling acquisition effects.

4. **Lesion Visualizer features**: all `tbp_lv_*` variables.
    - Potentially useful tabular predictors.
    - Analyze missing values, distributions, outliers, correlations and relationship with malignancy.

5. **Spatial/location variables**: `tbp_lv_location`, `tbp_lv_location_simple`, `tbp_lv_x`, `tbp_lv_y`, `tbp_lv_z`.
    - Useful to describe anatomical/spatial distribution.
    - Need to decide later whether they are clinically meaningful predictors or introduce unwanted acquisition/site effects.

   



    - ***age_approx***: approximate patient age at imaging. XXXX NA. Range: XXXX–XXXX.

    - ***sex***: patient sex. XXXX NA. Values:
        - XXXX = XXXX counts (XXXX%).
        - XXXX = XXXX counts (XXXX%).

    - ***anatom_site_general***: general anatomical site. XXXX NA. Values:
        - XXXX = XXXX counts (XXXX%).
        - XXXX = XXXX counts (XXXX%).

    - ***clin_size_long_diam_mm***: approximate maximum lesion diameter in mm. XXXX NA. Range: XXXX–XXXX.

    - ***image_type***: technical image type. XXXX NA. Values:
        - XXXX = XXXX counts (XXXX%).

    - ***tbp_tile_type***: lighting modality / tile type from 3D total body photography. XXXX NA. Values:
        - XXXX = XXXX counts (XXXX%).
        - XXXX = XXXX counts (XXXX%).

    - ***tbp_lv_location***: detailed anatomical location from Lesion Visualizer. XXXX NA. Values:
        - XXXX = XXXX counts (XXXX%).

    - ***tbp_lv_location_simple***: simplified anatomical location from Lesion Visualizer. XXXX NA. Values:
        - XXXX = XXXX counts (XXXX%).

    - ***tbp_lv_A, tbp_lv_B, tbp_lv_C, tbp_lv_H, tbp_lv_L***: color features inside the lesion. NA summary: XXXX.

    - ***tbp_lv_Aext, tbp_lv_Bext, tbp_lv_Cext, tbp_lv_Hext, tbp_lv_Lext***: color features outside the lesion. NA summary: XXXX.

    - ***tbp_lv_deltaA, tbp_lv_deltaB, tbp_lv_deltaL, tbp_lv_deltaLB, tbp_lv_deltaLBnorm***: contrast features between lesion and surrounding skin. NA summary: XXXX.

    - ***tbp_lv_areaMM2, tbp_lv_minorAxisMM, tbp_lv_perimeterMM***: size-related lesion features. NA summary: XXXX.

    - ***tbp_lv_area_perim_ratio, tbp_lv_eccentricity, tbp_lv_norm_border, tbp_lv_symm_2axis, tbp_lv_symm_2axis_angle***: shape, border and asymmetry features. NA summary: XXXX.

    - ***tbp_lv_color_std_mean, tbp_lv_norm_color, tbp_lv_radial_color_std_max, tbp_lv_stdL, tbp_lv_stdLExt***: color variability features. NA summary: XXXX.

    - ***tbp_lv_nevi_confidence***: model-derived nevus confidence score. XXXX NA. Range: XXXX–XXXX.

    - ***tbp_lv_x, tbp_lv_y, tbp_lv_z***: 3D spatial coordinates of the lesion. NA summary: XXXX. 

## final summary